## Импорты

In [1]:
CONFIG_NAME = "custom_CNN_RNN.yaml"

In [2]:
import sys
import os
from pathlib import Path

# Допустим, что ноутбук находится в той же директории, что и папка acoustic/
sys.path.insert(0, str(Path.cwd()))

import warnings
warnings.filterwarnings("ignore")

import logging
logging.getLogger("transformers.generation_utils").setLevel(logging.ERROR)

In [3]:
from acoustic.utils.config import load_config
from acoustic.dataset.load_dataset import load_and_prepare_dataset
from acoustic.models.load_model import build_model
from acoustic.training.load_metrics import load_metrics
from acoustic.training.callbacks import get_callback
from acoustic.training import get_trainer_class


## Загрузка конфигурации

In [4]:
CONFIG_PATH = f"acoustic/configs/{CONFIG_NAME}"

cfg = load_config(CONFIG_PATH, overrides=None)

print("Configuration loaded")

Configuration loaded


## Загрузка датасета

In [ ]:
dataset = load_and_prepare_dataset(cfg)

Applying filters:   0%|          | 0/2 [00:00<?, ?it/s]

Filter:   0%|          | 0/975995 [00:00<?, ? examples/s]

Exception ignored from cffi callback <function SoundFile._init_virtual_io.<locals>.vio_read at 0x7b1716897600>:
Traceback (most recent call last):
  File "/home/abonentvneseti/programming/github/STT_russian_lang/.venv/lib/python3.11/site-packages/soundfile.py", line 1241, in vio_read
    @_ffi.callback("sf_vio_read")

KeyboardInterrupt: 


## Инициализация модели

In [ ]:
model, processor, data_collator = build_model(cfg)

print("Model built")

## Создание и загрузка метрик, callbacks, trainer

In [ ]:
metrics_list = load_metrics(cfg['training']['metrics'])
print(f"Metrics: {cfg['training']['metrics']}")

callbacks = []
for cb_name in cfg['training']['callbacks']:
    callbacks.append(get_callback(cb_name))

In [ ]:

trainer_name = cfg['training'].get('trainer', 'BaseTrainer')
TrainerClass = get_trainer_class(trainer_name)

trainer = TrainerClass(
    cfg=cfg,
    model=model,
    processor=processor,
    train_dataset=dataset['train'],
    eval_dataset=dataset['validation'],
    metrics=metrics_list,
    callbacks=callbacks,
    data_collator=data_collator
)

## Обучение

In [ ]:
print("Starting training")
trainer.train()

## Проверка

In [ ]:
eval_dataset = dataset['validation']

print("Demo on validation examples")
import random
import torch
from acoustic.models import get_generate_method

builder_key = cfg['model']['builder']
generate_fn = get_generate_method(builder_key)

if eval_dataset and len(eval_dataset) > 0:
    indices = random.sample(range(len(eval_dataset)), min(3, len(eval_dataset)))
    device = next(model.parameters()).device
    model.eval()
    
    for i in indices:
        example = eval_dataset[i]
        audio_array = example["audio"]["array"]
        ref_text = example["sentence"]
        
        inputs = processor(audio_array, sampling_rate=16000, return_tensors="pt")
        input_key = "input_features" if "input_features" in inputs else "input_values"
        input_data = inputs[input_key].to(device)
        
        with torch.no_grad():
            predicted_ids = generate_fn(model, input_data, processor)
            pred_text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        
        print(f"\nExample {i+1}:")
        print(f" Reference: {ref_text}")
        print(f" Prediction: {pred_text}")
else:
    print("No validation dataset for demo.")